# EDA — Marche_Principal_Obligations_MERGED.csv

In [1]:
import pandas as pd
import numpy as np

df_obligations = pd.read_csv('D:/rouge_file/DATA_BULLETINS_CSV/Marche_Principal_Obligations_MERGED.csv', )


In [2]:
print(f'Shape : {df_obligations.shape}')

Shape : (16700, 18)


In [10]:
df_obligations.head(5)

,Type,Emetteur,Date Emission,Montant Emis,Montant à rembourser,Nominal,Maturité,Taux facial,Type amort.,Date dernier coupon,Cours Réf %,Instruments,Unnamed: 12,Cours du jour %,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,Type,NaN,NaN,KMAD,KMAD,KMAD,NaN,NaN,NaN,NaN,NaN,Code ISIN,Désignation,Statut,Prix,Coupon,Dem,Offre
1,EMPRUNTS OBLIGATAIRES A TAUX FIXE,ATTIJARIWFA BANK,17/12/14,"45 600,00","45 600,00","100,00",10,"4,75",FIN,22/12/21,"102,72",MA0000021651,22DEC2014 4.75% 10A 100K ATW,NT,"102,72","3,75",-,-
2,EMPRUNTS OBLIGATAIRES A TAUX FIXE,ATTIJARIWFA BANK,17/12/15,"64 600,00","64 600,00","100,00",10,"4,52",FIN,22/12/21,"103,42",MA0000021727,22DEC2015 4.52% 10A 100K ATW,NT,"103,42","3,57",-,-
3,EMPRUNTS OBLIGATAIRES A TAUX FIXE,ATTIJARIWFA BANK,17/12/15,"64 800,00","64 800,00","100,00",7,"4,13",FIN,22/12/21,"100,26",MA0000021735,22DEC2015 4.13% 7A 100K ATW,NT,"100,26","3,26",-,-
4,EMPRUNTS OBLIGATAIRES A TAUX FIXE,ATTIJARIWFA BANK,23/06/16,"100 000,00","100 000,00","100,00",10,"3,74",FIN,28/06/22,"101,15",MA0000021750,28JUN2016 3.74% 10A 100K ATW,NT,"101,15","1,02",-,-


## 1. Qualite des donnees

In [12]:
print('Valeurs manquantes :')
print(df_obligations.isnull().sum())


Valeurs manquantes :
Type                      0
Emetteur                851
Date Emission           851
Montant Emis              0
Montant à rembourser      0
Nominal                   0
Maturité                851
Taux facial             851
Type amort.             851
Date dernier coupon     852
Cours Réf %             851
Instruments               0
Unnamed: 12               0
Cours du jour %           0
Unnamed: 14               0
Unnamed: 15               0
Unnamed: 16               0
Unnamed: 17               0
dtype: int64


In [13]:
print(f'\nDoublons         : {df_obligations.duplicated().sum()}')



Doublons         : 908


In [14]:
print(f'Lignes parasites : {(df_obligations["Type"] == "Type").sum()}')


Lignes parasites : 679


In [15]:
print('\nValeurs colonne Type :')
print(df_obligations['Type'].value_counts())


Valeurs colonne Type :
Type
EMPRUNTS OBLIGATAIRES A TAUX FIXE         13449
EMPRUNTS OBLIGATAIRES A TAUX REVISABLE     2572
Type                                        679
Name: count, dtype: int64


## 2. Nettoyage

###  Renommage colonnes

In [16]:
df_obligations = df_obligations[df_obligations['Type'] != 'Type'].drop_duplicates().copy()

df_obligations = df_obligations.rename(columns={
    'Unnamed: 12' : 'Designation',
    'Unnamed: 14' : 'Prix',
    'Unnamed: 15' : 'Coupon',
    'Unnamed: 16' : 'Meilleur_Dem',
    'Unnamed: 17' : 'Meilleur_Offre',
    'Cours du jour %' : 'Statut',
})



### Statut : garder uniquement T/NT/S

In [17]:
statuts_valides = ['T', 'NT', 'S']
df_obligations['Statut'] = df_obligations['Statut'].where(df_obligations['Statut'].isin(statuts_valides), other=np.nan)


###  Conversion numerique

In [19]:
def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace(' ','').str.replace('\xa0','')
         .str.replace(',','.').str.replace('-','').str.strip(),
        errors='coerce'
    )

cols_num = ['Montant Emis','Montant à rembourser','Nominal','Maturité',
            'Taux facial','Cours Réf %','Prix','Coupon','Meilleur_Dem','Meilleur_Offre']
for col in cols_num:
    if col in df_obligations.columns:
        df_obligations[col] = to_num(df_obligations[col])

###  Conversion dates


In [20]:
for col in ['Date Emission','Date dernier coupon']:
    df_obligations[col] = pd.to_datetime(df_obligations[col], format='%d/%m/%y', errors='coerce')

print(f'Shape apres nettoyage : {df_obligations.shape}')
df_obligations.head(5)

Shape apres nettoyage : (15791, 18)


,Type,Emetteur,Date Emission,Montant Emis,Montant à rembourser,Nominal,Maturité,Taux facial,Type amort.,Date dernier coupon,Cours Réf %,Instruments,Designation,Statut,Prix,Coupon,Meilleur_Dem,Meilleur_Offre
1,EMPRUNTS OBLIGATAIRES A TAUX FIXE,ATTIJARIWFA BANK,2014-12-17,45600.0,45600.0,100.0,10.0,4.75,FIN,2021-12-22,102.72,MA0000021651,22DEC2014 4.75% 10A 100K ATW,NT,102.72,3.75,NaN,NaN
2,EMPRUNTS OBLIGATAIRES A TAUX FIXE,ATTIJARIWFA BANK,2015-12-17,64600.0,64600.0,100.0,10.0,4.52,FIN,2021-12-22,103.42,MA0000021727,22DEC2015 4.52% 10A 100K ATW,NT,103.42,3.57,NaN,NaN
3,EMPRUNTS OBLIGATAIRES A TAUX FIXE,ATTIJARIWFA BANK,2015-12-17,64800.0,64800.0,100.0,7.0,4.13,FIN,2021-12-22,100.26,MA0000021735,22DEC2015 4.13% 7A 100K ATW,NT,100.26,3.26,NaN,NaN
4,EMPRUNTS OBLIGATAIRES A TAUX FIXE,ATTIJARIWFA BANK,2016-06-23,100000.0,100000.0,100.0,10.0,3.74,FIN,2022-06-28,101.15,MA0000021750,28JUN2016 3.74% 10A 100K ATW,NT,101.15,1.02,NaN,NaN
5,EMPRUNTS OBLIGATAIRES A TAUX FIXE,ATTIJARIWFA BANK,2016-12-20,50000.0,50000.0,100.0,7.0,3.44,FIN,2021-12-23,100.67,MA0000021800,23DEC16 3.44% 7A 100K ATW,NT,100.67,2.70,NaN,NaN


## 3. Statistiques descriptives

In [21]:
df_obligations[['Montant Emis','Montant à rembourser','Nominal','Maturité','Taux facial','Cours Réf %','Prix','Coupon']].describe().round(2)

,Montant Emis,Montant à rembourser,Nominal,Maturité,Taux facial,Cours Réf %,Prix,Coupon
count,15790.00,15790.00,15790.00,13509.00,15790.00,15790.00,15772.00,15790.00
mean,156221.36,119571.96,85.32,10.32,4.31,100.21,100.21,2.27
std,133011.80,122005.07,31.68,2.41,0.61,1.40,1.40,1.31
min,5000.00,5000.00,6.67,7.00,2.22,93.22,93.22,0.00
25%,53300.00,45600.00,100.00,10.00,3.93,99.72,99.72,1.20
50%,100000.00,78900.00,100.00,10.00,4.10,100.34,100.34,2.27
75%,202900.00,183000.00,100.00,10.00,4.75,100.98,100.98,3.29
max,554200.00,554200.00,100.00,15.00,6.95,103.71,103.71,6.91


In [22]:
df_obligations.groupby('Type')[['Taux facial','Maturité','Cours Réf %','Prix']].mean().round(2)

,Taux facial,Maturité,Cours Réf %,Prix
Type,,,,
EMPRUNTS OBLIGATAIRES A TAUX FIXE,4.34,10.42,100.36,100.36
EMPRUNTS OBLIGATAIRES A TAUX REVISABLE,4.17,7.00,99.46,99.46


## 4. Analyse par Emetteur

In [24]:
print(f'Nombre d emetteurs distincts : {df_obligations["Emetteur"].nunique()}')
print()


Nombre d emetteurs distincts : 25



In [25]:
print('Top 10 emetteurs par nombre d obligations :')
print(df_obligations['Emetteur'].value_counts().head(10).to_string())
print()

Top 10 emetteurs par nombre d obligations :
Emetteur
ATTIJARIWAFA BANK     3191
CREDIT AGRICOLE DU    1413
CREDIT AGRICOLE MA    1377
CDM                   1358
BANK OF AFRICA         924
OCP SA                 732
ATTIJARIWFA BANK       650
CIH                    630
SGMB                   629
ONCF                   628



In [26]:
print('Top 10 emetteurs par montant emis total (KMAD) :')
top_montant = df_obligations.groupby('Emetteur')['Montant Emis'].sum().sort_values(ascending=False).head(10)
print(top_montant.round(0).to_string())

Top 10 emetteurs par montant emis total (KMAD) :
Emetteur
OCP SA                 269815200.0
ATTIJARIWAFA BANK      256257400.0
CREDIT AGRICOLE DU     248622500.0
CREDIT AGRICOLE MA     231644000.0
OCP                    230743600.0
ONCF                   201336800.0
OFFICE NATIONAL DES    173255600.0
CDM                    171447500.0
LYDEC                  103682800.0
SGMB                    90976000.0


## 5. Analyse des Taux

In [27]:
print('Taux facial par type d emprunt :')
print(df_obligations.groupby('Type')['Taux facial'].describe().round(4).to_string())
print()


Taux facial par type d emprunt :
                                          count    mean     std   min   25%   50%   75%   max
Type                                                                                         
EMPRUNTS OBLIGATAIRES A TAUX FIXE       13223.0  4.3396  0.6494  3.32  3.93  4.10  4.77  6.95
EMPRUNTS OBLIGATAIRES A TAUX REVISABLE   2567.0  4.1702  0.3492  2.22  4.03  4.07  4.69  4.69



In [28]:
print('Distribution Type amortissement :')
print(df_obligations['Type amort.'].value_counts().to_string())
print('  FIN = Amortissement in fine | SEA = Amortissement par series egales')

Distribution Type amortissement :
Type amort.
FIN    12949
SEA     2841
  FIN = Amortissement in fine | SEA = Amortissement par series egales


In [29]:
print('Top 10 taux faciaux les plus eleves :')
top_taux = df_obligations.nlargest(10,'Taux facial')[['Emetteur','Designation','Taux facial','Maturité','Type']]
print(top_taux.to_string(index=False))
print()


Top 10 taux faciaux les plus eleves :
      Emetteur                  Designation  Taux facial  Maturité                              Type
BANK OF AFRICA 15OCT2008 6.95% IND 100K BOA         6.95       NaN EMPRUNTS OBLIGATAIRES A TAUX FIXE
BANK OF AFRICA 15OCT2008 6.95% IND 100K BOA         6.95       NaN EMPRUNTS OBLIGATAIRES A TAUX FIXE
BANK OF AFRICA 15OCT2008 6.95% IND 100K BOA         6.95       NaN EMPRUNTS OBLIGATAIRES A TAUX FIXE
BANK OF AFRICA 15OCT2008 6.95% IND 100K BOA         6.95       NaN EMPRUNTS OBLIGATAIRES A TAUX FIXE
BANK OF AFRICA 15OCT2008 6.95% IND 100K BOA         6.95       NaN EMPRUNTS OBLIGATAIRES A TAUX FIXE
BANK OF AFRICA 15OCT2008 6.95% IND 100K BOA         6.95       NaN EMPRUNTS OBLIGATAIRES A TAUX FIXE
BANK OF AFRICA 15OCT2008 6.95% IND 100K BOA         6.95       NaN EMPRUNTS OBLIGATAIRES A TAUX FIXE
BANK OF AFRICA 15OCT2008 6.95% IND 100K BOA         6.95       NaN EMPRUNTS OBLIGATAIRES A TAUX FIXE
BANK OF AFRICA 15OCT2008 6.95% IND 100K BOA         6

In [30]:
print('Top 10 taux faciaux les plus bas :')
low_taux = df_obligations.nsmallest(10,'Taux facial')[['Emetteur','Designation','Taux facial','Maturité','Type']]
print(low_taux.to_string(index=False))

Top 10 taux faciaux les plus bas :
         Emetteur                 Designation  Taux facial  Maturité                                   Type
ATTIJARIWAFA BANK 28DEC2017 2.22% 7A 100K ATW         2.22       7.0 EMPRUNTS OBLIGATAIRES A TAUX REVISABLE
 ATTIJARIWFA BANK 29JUN2018 3.32% 7A 100K ATW         3.32       7.0      EMPRUNTS OBLIGATAIRES A TAUX FIXE
ATTIJARIWAFA BANK 29JUN2018 3.32% 7A 100K ATW         3.32       7.0      EMPRUNTS OBLIGATAIRES A TAUX FIXE
ATTIJARIWAFA BANK 29JUN2018 3.32% 7A 100K ATW         3.32       7.0      EMPRUNTS OBLIGATAIRES A TAUX FIXE
ATTIJARIWAFA BANK 29JUN2018 3.32% 7A 100K ATW         3.32       7.0      EMPRUNTS OBLIGATAIRES A TAUX FIXE
ATTIJARIWAFA BANK 29JUN2018 3.32% 7A 100K ATW         3.32       7.0      EMPRUNTS OBLIGATAIRES A TAUX FIXE
ATTIJARIWAFA BANK 29JUN2018 3.32% 7A 100K ATW         3.32       7.0      EMPRUNTS OBLIGATAIRES A TAUX FIXE
ATTIJARIWAFA BANK 29JUN2018 3.32% 7A 100K ATW         3.32       7.0      EMPRUNTS OBLIGATAIRES A TAU

## 6. Analyse des Maturites

In [31]:
print('Distribution des maturites (en annees) :')
print(df_obligations['Maturité'].value_counts().sort_index().to_string())
print()


Distribution des maturites (en annees) :
Maturité
7.0     2437
10.0    8746
15.0    2326



In [32]:
print(f'Maturite moyenne   : {df_obligations["Maturité"].mean():.1f} ans')

Maturite moyenne   : 10.3 ans


In [33]:
print(f'Maturite max       : {df_obligations["Maturité"].max():.0f} ans')

Maturite max       : 15 ans


In [34]:
print(f'Maturite min       : {df_obligations["Maturité"].min():.0f} ans')

Maturite min       : 7 ans


## 7. Analyse des Cours

In [35]:
print('Stats cours de reference et prix :')
print(df_obligations[['Cours Réf %','Prix','Coupon']].describe().round(2).to_string())
print()


Stats cours de reference et prix :
       Cours Réf %      Prix    Coupon
count     15790.00  15772.00  15790.00
mean        100.21    100.21      2.27
std           1.40      1.40      1.31
min          93.22     93.22      0.00
25%          99.72     99.72      1.20
50%         100.34    100.34      2.27
75%         100.98    100.98      3.29
max         103.71    103.71      6.91



### Ecart cours ref vs prix

In [36]:
df_obligations['Ecart_Cours_Prix'] = df_obligations['Prix'] - df_obligations['Cours Réf %']
print(f'Ecart moyen Cours Ref vs Prix : {df_obligations["Ecart_Cours_Prix"].mean():.4f}%')
print(f'Ecart max                     : {df_obligations["Ecart_Cours_Prix"].max():.4f}%')
print(f'Ecart min                     : {df_obligations["Ecart_Cours_Prix"].min():.4f}%')

Ecart moyen Cours Ref vs Prix : 0.0000%
Ecart max                     : 0.4900%
Ecart min                     : -0.2300%


## 8. Outliers

In [37]:
cols_check = [col for col in ['Montant Emis','Taux facial','Maturité','Cours Réf %','Prix']
              if df_obligations[col].dtype in ['float64','int64']]
for col in cols_check:
    Q1, Q3 = df_obligations[col].quantile(0.25), df_obligations[col].quantile(0.75)
    IQR    = Q3 - Q1
    out    = df_obligations[(df_obligations[col] < Q1-1.5*IQR) | (df_obligations[col] > Q3+1.5*IQR)]
    print(f'{col:25s} : {len(out):,} outliers ({len(out)/len(df_obligations)*100:.1f}%)')

Montant Emis              : 679 outliers (4.3%)
Taux facial               : 126 outliers (0.8%)
Maturité                  : 4,763 outliers (30.2%)
Cours Réf %               : 1,093 outliers (6.9%)
Prix                      : 1,093 outliers (6.9%)


## 9. Correlation

In [38]:
cols_corr = [c for c in ['Montant Emis','Taux facial','Maturité','Cours Réf %','Prix','Coupon']
             if df_obligations[c].dtype in ['float64','int64']]
df_obligations[cols_corr].corr().round(4)

,Montant Emis,Taux facial,Maturité,Cours Réf %,Prix,Coupon
Montant Emis,1.0000,0.3064,0.5836,0.1507,0.1508,0.0630
Taux facial,0.3064,1.0000,0.8526,0.1801,0.1804,0.2936
Maturité,0.5836,0.8526,1.0000,0.2462,0.2463,0.1810
Cours Réf %,0.1507,0.1801,0.2462,1.0000,1.0000,0.0024
Prix,0.1508,0.1804,0.2463,1.0000,1.0000,0.0022
Coupon,0.0630,0.2936,0.1810,0.0024,0.0022,1.0000


## 10. Export

In [3]:
print('Colonnes avant renommage:')
print(list(df_obligations.columns))
 
df_obligations = df_obligations.rename(columns={
    'Unnamed: 12'    : 'Designation',
    'Cours du jour %': 'Statut',
    'Unnamed: 14'    : 'Prix',
    'Unnamed: 15'    : 'Coupon',
    'Unnamed: 16'    : 'Meilleur_Demande',
    'Unnamed: 17'    : 'Meilleur_Offre',
})
 
print()
print('Colonnes apres renommage:')
print(list(df_obligations.columns))
 
df_obligations.to_csv('Obligations_RENAMED.csv', index=False, encoding='utf-8')
print()
print(f'Exporte : Obligations_RENAMED.csv — {len(df_obligations):,} lignes')
 

Colonnes avant renommage:
['Type', 'Emetteur', 'Date Emission', 'Montant Emis', 'Montant à rembourser', 'Nominal', 'Maturité', 'Taux facial', 'Type amort.', 'Date dernier coupon', 'Cours Réf %', 'Instruments', 'Unnamed: 12', 'Cours du jour %', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17']

Colonnes apres renommage:
['Type', 'Emetteur', 'Date Emission', 'Montant Emis', 'Montant à rembourser', 'Nominal', 'Maturité', 'Taux facial', 'Type amort.', 'Date dernier coupon', 'Cours Réf %', 'Instruments', 'Designation', 'Statut', 'Prix', 'Coupon', 'Meilleur_Demande', 'Meilleur_Offre']

Exporte : Obligations_RENAMED.csv — 16,700 lignes
